<a href="https://colab.research.google.com/github/mahamoorthi22007/AI-Internship/blob/main/Resume_Analysis_using_ChromaDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb sentence-transformers pypdf groq

In [ ]:
from pypdf import PdfReader

def extract_text(pdf_path):
    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text

In [ ]:
resume1 = extract_text("/content/MaansaResume.pdf")
resume2 = extract_text("/content/MahaResume.pdf")

In [ ]:
def chunk_text(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])

    return chunks

In [ ]:
chunks1 = chunk_text(resume1)
chunks2 = chunk_text(resume2)

In [ ]:
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="resume_collection"
)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
all_chunks = []
all_ids = []
all_metadatas = []

counter = 0

for chunk in chunks1:
    all_chunks.append(chunk)
    all_ids.append(str(counter))

    all_metadatas.append({
        "candidate":"Resume1"
    })

    counter += 1

for chunk in chunks2:
    all_chunks.append(chunk)
    all_ids.append(str(counter))

    all_metadatas.append({
        "candidate":"Resume2"
    })

    counter += 1


In [ ]:
embeddings = model.encode(all_chunks).tolist()

In [ ]:
collection.add(
    ids=all_ids,
    embeddings=embeddings,
    documents=all_chunks,
    metadatas=all_metadatas
)

In [ ]:
from groq import Groq

groq_client = Groq(
    api_key="YOUR_GROQ_API_KEY"
)

In [ ]:
def ask_resume(question):

    query_embedding = model.encode(
        [question]
    ).tolist()[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    context = "\n".join(
        results["documents"][0]
    )

    prompt = f"""
You are a resume assistant.

Answer only from the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content